# Notebook 2 of 3 — Preprocessing

**Research Project:** Lightweight Machine Learning-Based Cybersecurity Threat Detection for Resource-Constrained Hardware
**Author:** Anush Jindal

This notebook is Stage 3 ("Preprocessing") of the project. Using what you learned in `01_dataset_exploration.ipynb`, we now:
1. Clean missing values, infinite values, and duplicates.
2. Decide which identifier columns to remove, and why (data leakage).
3. Split into train/test **before** fitting any preprocessing — this is the single most important rule in this notebook.

**We do not train a model in this notebook.** The output of this notebook is a clean, split, leakage-safe dataset — that's it.

## 0. Why train/test split comes *before* cleaning decisions matter

Read this before writing any code.

**Data leakage** happens when information from the test set (data the model should never see during training) influences training in any way — even indirectly. If that happens, your evaluation results will look better than the model will actually perform on truly new data.

The correct order is:

```
Raw dataset
   ↓
Basic cleaning (things that don't use any statistics computed from the data, e.g.
                 dropping rows with infinite values, dropping duplicate rows)
   ↓
Train/Test Split
   ↓
Fit preprocessing on TRAINING data only (e.g. scalers, encoders, imputers)
   ↓
Transform train (using what was fit on train)
   ↓
Transform test (using the SAME fitted transformer from train — never re-fit on test)
```

**NOT this:**
```
Entire dataset → preprocess everything together → THEN split   ⟵ WRONG, leaks test info into training
```

Concretely in this notebook: dropping rows with missing/infinite values is fine to do *before* splitting, because it doesn't compute any statistic from the data (it's just a row filter, not something "learned" from the data). But anything that *learns* something from the data's distribution — scaling, encoding categories, imputing with a mean/median, selecting "important" features by correlation with the label — must be fit on the training set only, after the split.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")

In [ ]:
# Same file you used in 01_dataset_exploration.ipynb
FILENAME = "Wednesday-workingHours.pcap_ISCX.csv"
LABEL_COLUMN = "Label"

df = pd.read_csv(FILENAME)
df.columns = df.columns.str.strip()  # same whitespace fix as notebook 1

print("Loaded shape:", df.shape)

## 2. Handle infinite values

CICIDS2017's rate-based columns (e.g. `Flow Bytes/s`, `Flow Packets/s`) can be `inf` when a flow's duration was 0 (division by zero). `isnull()` does not detect `inf` — we have to check separately, as you saw in Notebook 1.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

inf_mask = np.isinf(df[numeric_cols]).any(axis=1)
print(f"Rows containing at least one infinite value: {inf_mask.sum()} out of {len(df)}")
print(f"That's {inf_mask.sum() / len(df) * 100:.3f}% of rows")

**Decision:** if this is a small fraction of rows (a common outcome — often well under 1%), the simplest and safest option is to drop them, since imputing a "reasonable" value for a mathematically undefined rate (division by zero) is not obviously meaningful. If it turns out to be a large fraction in your file, stop and reconsider (post in `experiments/experiment_notes.md` what you found and why dropping might not be appropriate) rather than silently dropping a big chunk of your data.

This is a row filter based on the raw values themselves, not a statistic learned from the dataset's distribution — so it's safe to do before the train/test split.

In [ ]:
rows_before = len(df)
df = df[~inf_mask].copy()
rows_after = len(df)

print(f"Dropped {rows_before - rows_after} rows containing infinite values.")
print(f"Shape is now: {df.shape}")

## 3. Handle missing values

Re-check for `NaN`s now that infinite-value rows are gone.

In [ ]:
missing = df.isnull().sum()
missing_only = missing[missing > 0].sort_values(ascending=False)
print(f"Columns still containing missing values: {len(missing_only)}")
missing_only

**Decision:** for the same reason as infinite values — if missing values only affect a small number of rows, dropping those rows is the simplest defensible choice for this baseline pipeline (rather than guessing at an imputed value for a network-flow measurement). Document what you actually found before running the cell below; if a specific column is missing a *large* fraction of its values, dropping the whole column may be more appropriate than dropping rows — think about it rather than applying this blindly.

In [ ]:
rows_before = len(df)
df = df.dropna().copy()
rows_after = len(df)

print(f"Dropped {rows_before - rows_after} rows containing missing values.")
print(f"Shape is now: {df.shape}")

## 4. Handle duplicate rows

In [ ]:
rows_before = len(df)
df = df.drop_duplicates().copy()
rows_after = len(df)

print(f"Dropped {rows_before - rows_after} duplicate rows.")
print(f"Shape is now: {df.shape}")

## 5. Remove identifier columns (data leakage risk)

Some columns are identifiers rather than behavioural measurements — keeping them risks the model "memorizing" specific IPs, ports, timestamps, or flow IDs seen in training, instead of learning general attack *behaviour*. That would fall apart against a real, previously-unseen attacker.

**Before running the cell below:** go check `df.columns` from Notebook 1 against this list. Not every dataset version has all of these columns, and some may be spelled slightly differently — adjust the list to match your actual columns, and fill in `research/feature_documentation_template.md` with your Keep/Remove reasoning for **every** column, not just these.

In [ ]:
# Adjust this list to match the EXACT column names in your file (check df.columns first)
IDENTIFIER_COLUMNS_TO_REMOVE = [
    "Flow ID",
    "Source IP",
    "Destination IP",
    "Timestamp",
]

existing_to_remove = [c for c in IDENTIFIER_COLUMNS_TO_REMOVE if c in df.columns]
missing_from_df = [c for c in IDENTIFIER_COLUMNS_TO_REMOVE if c not in df.columns]

print("Will remove:", existing_to_remove)
if missing_from_df:
    print("Not found in this file (skipping):", missing_from_df)

df = df.drop(columns=existing_to_remove)
print("\nShape is now:", df.shape)

**Note on Source Port / Destination Port:** these are intentionally *not* in the removal list above. Unlike a raw IP address or flow ID, a destination port (e.g. 22, 80, 443) can carry genuine behavioural signal (which service is being targeted). Whether to keep, drop, or transform them (e.g. bucket into "well-known" vs "ephemeral" ports) is a real feature-engineering decision — make it explicitly in Notebook 3, not silently here.

## 6. Separate features (X) and target label (y)

In [ ]:
X = df.drop(columns=[LABEL_COLUMN])
y = df[LABEL_COLUMN]

print("X shape:", X.shape)
print("y shape:", y.shape)

## 7. Train/test split — THE step where leakage is usually introduced or avoided

We split now, before fitting any scaler/encoder. `stratify=y` keeps the class proportions roughly the same in both the train and test sets — important here because BENIGN vastly outnumbers most attack classes, and a plain random split could leave very few (or zero) examples of a rare attack type in the test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

In [ ]:
print("Class balance in y_train:")
print((y_train.value_counts(normalize=True) * 100).round(2))
print("\nClass balance in y_test:")
print((y_test.value_counts(normalize=True) * 100).round(2))

**What to check:** the percentages for each class should look very similar between `y_train` and `y_test` (that's `stratify=y` working correctly). If a class is so rare that `train_test_split` raises an error about it (`stratify` requires at least 2 members per class), note that in `experiments/experiment_notes.md` — it's a real, worth-documenting limitation of working with a single day's file.

## 8. Any further preprocessing (scaling, encoding) must fit on X_train only

This notebook stops here deliberately — actual feature engineering (deciding which columns to transform, scale, bucket, or engineer from others) is Notebook 3's job. But the rule to carry forward is:

```python
# CORRECT pattern for any future scaler/encoder:
scaler = StandardScaler()
scaler.fit(X_train)              # learn mean/std from TRAINING data only
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)   # apply the SAME fitted scaler, never re-fit on test
```

Never call `.fit()` or `.fit_transform()` on `X_test` — only `.transform()`.

## 9. Save your cleaned, split data for the next notebook

Save these to CSV files inside `data/` so Notebook 3 can load them directly instead of re-doing all this cleaning. Remember: these processed files should **also** not be committed to GitHub — `data/*.csv` is already excluded by `.gitignore`, so this is automatic, but don't override it.

In [ ]:
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Saved X_train.csv, X_test.csv, y_train.csv, y_test.csv")
print("If working locally, move these four files into the data/ folder.")
print("If working in Colab, download them (folder icon > right-click each file > Download),")
print("or save them to your mounted Google Drive so Notebook 3 can load them next time.")

## 10. Summary — record your observations

Add an entry to `experiments/experiment_notes.md` covering:

- How many rows were dropped for infinite values, missing values, and duplicates (and as a % of the original)?
- Which identifier columns did you remove, and does that match `research/feature_documentation_template.md`?
- Final `X_train` / `X_test` shapes.
- Did stratified splitting work cleanly, or did any class cause a problem?

Once this is recorded, move on to `03_FEATURE_ENGINEERING_GUIDE.md`.